# NB00 — data integrity and safe split

**One custom architecture • three seeds • 60 epochs • no pretrained weights.**

Use **CPU / Accelerator None**, Internet ON, Kaggle secret **HF_TOKEN** with write permission to **Shanmuk4622/tyre-wear-study**.
Attach [Tire Dataset Prepared phase2](https://www.kaggle.com/datasets/shanmuk4622/tire-dataset-prepared-phase2).
No repository/dataset ZIP upload is needed in addition to that input. Run All in order, one copy of NB01 only.

The dataset is 570 images of 12 tyres; the frozen 386/81/103 split holds out only old low/high tyres. All known mid/video material is training-only. Classification is a mileage proxy. Boundaries and masks do not measure physical alignment or tread depth.

This cell sequence verifies all source files and publishes a compact report. Its CPU runtime anchor is separate from training.

**Recovery:** local checkpoint after each completed optimizer update; consistent HF bundles every 30 minutes, at seed/cell completion and catchable Stop/error. Restore uses the same RUN_TAG/source/environment. Hard kernel death can lose work since the last verified upload. Watch for `HF snapshot verified` and retain the printed revision.

The exact additive maps explain class logits at the evidence layer. They do not prove physical wear reasoning or pixel-level causality; final output includes background/occlusion diagnostics and failures.


## 1. Bootstrap and secrets
Do not paste token values into cells. The embedded payload is generated from readable Phase 3 source.

In [1]:
# Install only the tested Hub/locking tools; retain Kaggle's CUDA PyTorch.
import base64, gzip, hashlib, importlib.metadata, json, os, subprocess, sys
from pathlib import Path
for package, version in [('huggingface_hub', '0.36.2'), ('filelock', '3.29.5')]:
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', package+'=='+version])
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
RUNTIME = Path('/kaggle/working/treadone_runtime')
RUNTIME.mkdir(parents=True, exist_ok=True)
payload = json.loads(gzip.decompress(base64.b64decode('...')))
for name, encoded in payload.items():
    assert Path(name).name == name
    value = base64.b64decode(encoded)
    destination = RUNTIME/name
    temporary = destination.with_suffix(destination.suffix+'.tmp')
    temporary.write_bytes(value)
    temporary.replace(destination)
bundle = json.loads((RUNTIME/'BUNDLE.json').read_text())
for name, expected in bundle['sources_sha256'].items():
    assert hashlib.sha256((RUNTIME/name).read_bytes()).hexdigest() == expected, name
print('Verified', len(bundle['sources_sha256']), 'runtime files. HF_TOKEN loaded without displaying it.')


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.36.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


Verified 9 runtime files. HF_TOKEN loaded without displaying it.


## 2. Configure and define the safe launcher

In [2]:
# Keep these unchanged when resuming the same experiment.
DATA_ROOT = ''  # Auto-detect VERSION.json in the attached Kaggle input.
RUN_TAG = 'treadone-r1'
TAKE_OVER = False  # True ONCE only after the older copy is stopped and has flushed.
ALLOW_SINGLE_GPU = False  # Default training expects T4 x2; no silent fallback.
WORK = Path('/kaggle/working/treadone_preflight')
WORK.mkdir(parents=True, exist_ok=True)
arguments = ['preflight', '--root', DATA_ROOT, '--work', str(WORK), '--tag', RUN_TAG]
if TAKE_OVER:
    arguments.append('--take-over')
if ALLOW_SINGLE_GPU:
    arguments.append('--allow-single-gpu')


In [3]:
import signal, time
def launch(arguments):
    process = subprocess.Popen([sys.executable, '-B', '-u', str(RUNTIME/'run.py'), *arguments],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
    except KeyboardInterrupt:
        print('Stop requested. Wait for the checkpoint/HF flush; a second Stop or kernel kill can interrupt cleanup.', flush=True)
        process.send_signal(signal.SIGINT)
        for line in process.stdout:
            print(line, end='', flush=True)
        process.wait()
        raise
    if status:
        raise RuntimeError('Run stopped with an error. Read the retained console/checkpoint and last verified HF revision; rerun unchanged to resume.')


## 3. Execute
Stop once and wait for the verified HF flush. A server backoff or network outage may delay it.

In [4]:
launch(arguments)

Hash checked 500/2882 files
Hash checked 1000/2882 files
Hash checked 1500/2882 files
Hash checked 2000/2882 files
Hash checked 2500/2882 files
{
  "integrity": "PASS",
  "checksummed_files": 2882,
  "image_and_mask_sets": 570,
  "domains": {
    "original_photo": 418,
    "new_video_frame": 152
  },
  "conflict_pixels_explicitly_ignored": 172439,
  "source_package_ready": true,
  "training_allowed": false,
  "next_steps": [
    "Resolve exact original/video tyre identities and freeze splits",
    "Resolve label conflicts for YOLO or other trainers without ignore support",
    "Review derived geometry proposals",
    "Run the forthcoming GPU/resume preflight"
  ]
}
HF snapshot verified: 1168e2e27311e39931ca2ccc87d214df4d6721e7 {}


In [5]:
print(json.dumps(json.loads((WORK/'DATA_PREFLIGHT.json').read_text()), indent=2))
print('Preflight complete. Continue with NB01 in a dual-T4 session; no training was performed here.')

{
  "geometry_counts": {
    "test": 25,
    "train": 222,
    "validation": 25
  },
  "groups": {
    "test": [
      "mileage_000100_005000__session_002",
      "mileage_100000_plus__session_005"
    ],
    "train": [
      "mileage_040000__session_001",
      "mileage_070000__session_001",
      "mileage_090000__session_001",
      "mileage_100000_plus__session_001",
      "mileage_100000_plus__session_002",
      "mileage_100000_plus__session_003",
      "mileage_100000_plus__session_004",
      "new_tire__session_001"
    ],
    "validation": [
      "mileage_000100_005000__session_001",
      "mileage_100000_plus__session_006"
    ]
  },
  "limitations": {
    "evaluation": "Held-out old low/high tyres only. No unseen-mid, unseen-video, three-class-generalisation or safety claim.",
    "geometry": "New polygon-derived points are weak training targets with weight 0.25; validation/test use only existing human points.",
    "identity": "All three original mid tyres and all new frame